# Logistic Regression from Scratch

Classification with linear regression + sigmoid. Understanding binary classification, cross-entropy loss, and decision boundaries.

## The Model: P(y=1|x) = σ(w·x + b)

Sigmoid function maps linear output to probability (0-1). Threshold at 0.5 for binary classification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## Sigmoid Function

σ(z) = 1/(1 + e^(-z)). Maps any real number to (0,1) range.

In [ ]:
def sigmoid(z):
    """Sigmoid activation function"""
    return 1 / (1 + np.exp(-z))

# Plot sigmoid
z = np.linspace(-10, 10, 100)
sigma_z = sigmoid(z)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(z, sigma_z, 'b-', linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.7, label='Decision threshold')
plt.axvline(x=0, color='g', linestyle='--', alpha=0.7, label='z=0')
plt.xlabel('z = w·x + b')
plt.ylabel('σ(z)')
plt.title('Sigmoid Function')
plt.legend()
plt.grid(True, alpha=0.3)

# Derivative of sigmoid
plt.subplot(1, 2, 2)
sigmoid_derivative = sigma_z * (1 - sigma_z)
plt.plot(z, sigmoid_derivative, 'g-', linewidth=2)
plt.xlabel('z')
plt.ylabel('σ′(z)')
plt.title('Sigmoid Derivative')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Sigmoid Properties:")
print("- σ(0) = 0.5")
print("- σ(z) ∈ (0,1) for all z")
print("- σ(-z) = 1 - σ(z)")
print("- Derivative: σ′(z) = σ(z)·(1-σ(z))")

## Logistic Regression Implementation

Cross-entropy loss: L = -Σ[y·log(ŷ) + (1-y)·log(1-ŷ)]

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.cost_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for i in range(self.n_iterations):
            # Forward pass
            linear_model = np.dot(X, self.weights) + self.bias
            y_pred = sigmoid(linear_model)
            
            # Compute gradients
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Track cost
            cost = self._compute_cost(y, y_pred)
            self.cost_history.append(cost)
            
            if i % 100 == 0:
                print(f"Iteration {i}: Cost = {cost:.6f}")
        
        return self
    
    def _compute_cost(self, y_true, y_pred):
        """Binary cross-entropy loss"""
        n_samples = len(y_true)
        # Add small epsilon to avoid log(0)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        cost = -(1/n_samples) * np.sum(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )
        return cost
    
    def predict_proba(self, X):
        """Predict probabilities"""
        linear_model = np.dot(X, self.weights) + self.bias
        return sigmoid(linear_model)
    
    def predict(self, X, threshold=0.5):
        """Predict classes"""
        y_pred_proba = self.predict_proba(X)
        return (y_pred_proba >= threshold).astype(int)
    
    def get_coefficients(self):
        return self.weights, self.bias

## Training and Decision Boundary

Visualising how logistic regression separates classes.

In [ ]:
# Generate binary classification data
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                          n_clusters_per_class=1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = LogisticRegressionScratch(learning_rate=0.1, n_iterations=1000)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

# Results
weights, bias = model.get_coefficients()
accuracy = accuracy_score(y_test, y_pred)
print(f"Weights: {weights}")
print(f"Bias: {bias:.3f}")
print(f"Accuracy: {accuracy:.3f}")

# Plot decision boundary
def plot_decision_boundary(X, y, model, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='coolwarm')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title(title)
    plt.grid(True, alpha=0.3)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plot_decision_boundary(X_train, y_train, model, 'Training Data: Decision Boundary')

plt.subplot(1, 2, 2)
plot_decision_boundary(X_test, y_test, model, 'Test Data: Decision Boundary')

plt.tight_layout()
plt.show()

# Cost history
plt.figure(figsize=(8, 5))
plt.plot(model.cost_history)
plt.xlabel('Iteration')
plt.ylabel('Cost (Cross-entropy)')
plt.title('Logistic Regression: Cost vs Iterations')
plt.grid(True, alpha=0.3)
plt.show()

## Threshold Tuning

Different thresholds change precision/recall balance.

In [ ]:
# Test different thresholds
thresholds = np.linspace(0.1, 0.9, 9)
accuracies = []
precisions = []
recalls = []

for threshold in thresholds:
    y_pred_thresh = model.predict(X_test, threshold=threshold)
    cm = confusion_matrix(y_test, y_pred_thresh)
    
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        acc = (tp + tn) / (tp + tn + fp + fn)
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        accuracies.append(acc)
        precisions.append(prec)
        recalls.append(rec)
    else:
        accuracies.append(0)
        precisions.append(0)
        recalls.append(0)

# Plot threshold analysis
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(thresholds, accuracies, 'b-o', linewidth=2, label='Accuracy')
plt.xlabel('Threshold')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Threshold')
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
plt.plot(thresholds, precisions, 'g-o', linewidth=2, label='Precision')
plt.plot(thresholds, recalls, 'r-o', linewidth=2, label='Recall')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)

# Precision-Recall curve
plt.subplot(2, 2, 3)
plt.plot(recalls, precisions, 'b-', linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True, alpha=0.3)

# Confusion matrix at optimal threshold
optimal_threshold = thresholds[np.argmax(accuracies)]
y_pred_optimal = model.predict(X_test, threshold=optimal_threshold)
cm_optimal = confusion_matrix(y_test, y_pred_optimal)

plt.subplot(2, 2, 4)
sns.heatmap(cm_optimal, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title(f'Confusion Matrix (threshold={optimal_threshold:.1f})')

plt.tight_layout()
plt.show()

print(f"Optimal threshold: {optimal_threshold:.1f}")
print(f"Accuracy at optimal threshold: {max(accuracies):.3f}")
print("\nThreshold Guidelines:")
print("- 0.5: Balanced (default)")
print("- <0.5: More sensitive (catches more positives)")
print("- >0.5: More specific (fewer false positives)")

## Regularisation

L2 regularisation prevents overfitting in logistic regression.

In [ ]:
class LogisticRegressionRegularised:
    def __init__(self, learning_rate=0.01, n_iterations=1000, lambda_reg=0.1):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.lambda_reg = lambda_reg
        self.weights = None
        self.bias = None
        self.cost_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for i in range(self.n_iterations):
            linear_model = np.dot(X, self.weights) + self.bias
            y_pred = sigmoid(linear_model)
            
            # Gradients with L2 regularisation
            dw = (1/n_samples) * (np.dot(X.T, (y_pred - y)) + self.lambda_reg * self.weights)
            db = (1/n_samples) * np.sum(y_pred - y)
            
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            cost = self._compute_cost(y, y_pred)
            self.cost_history.append(cost)
        
        return self
    
    def _compute_cost(self, y_true, y_pred):
        n_samples = len(y_true)
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        
        # Cross-entropy + L2 regularisation
        ce_loss = -(1/n_samples) * np.sum(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )
        reg_term = (self.lambda_reg/(2*n_samples)) * np.sum(self.weights**2)
        
        return ce_loss + reg_term
    
    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return sigmoid(linear_model)
    
    def predict(self, X, threshold=0.5):
        y_pred_proba = self.predict_proba(X)
        return (y_pred_proba >= threshold).astype(int)

# Test regularisation
lambdas = [0, 0.01, 0.1, 1.0]
colors = ['red', 'orange', 'green', 'blue']

plt.figure(figsize=(12, 8))

for lambda_reg, color in zip(lambdas, colors):
    model_reg = LogisticRegressionRegularised(learning_rate=0.1, lambda_reg=lambda_reg)
    model_reg.fit(X_train, y_train)
    
    y_pred_reg = model_reg.predict(X_test)
    acc = accuracy_score(y_test, y_pred_reg)
    
    plt.subplot(2, 2, 1)
    plt.plot(model_reg.cost_history, color=color, label=f'λ={lambda_reg}')
    
    plt.subplot(2, 2, 2)
    plt.scatter(X_test[:, 0], X_test[:, 1], c=y_pred_reg, alpha=0.6, 
                edgecolors='k', cmap='coolwarm')
    plt.title(f'Predictions (λ={lambda_reg}, acc={acc:.3f})')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')

plt.subplot(2, 2, 1)
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Regularisation: Cost vs Iterations')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Regularisation in Logistic Regression:")
print("- Prevents overfitting by penalising large weights")
print("- λ=0: No regularisation")
print("- λ>0: Smaller weights, smoother decision boundaries")
print("- Higher λ = stronger regularisation")

## Multiclass Extension (Softmax)

One-vs-rest or softmax for multiple classes.

In [ ]:
def softmax(z):
    """Softmax for multiclass"""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # Numerical stability
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# Generate 3-class data
X_multi, y_multi = make_classification(n_samples=300, n_features=2, n_classes=3, 
                                       n_redundant=0, n_clusters_per_class=1, random_state=42)

# One-hot encode labels
def one_hot_encode(y, n_classes):
    return np.eye(n_classes)[y]

y_multi_onehot = one_hot_encode(y_multi, 3)

# Simple softmax regression
class SoftmaxRegression:
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        n_classes = y.shape[1]
        
        self.weights = np.zeros((n_features, n_classes))
        self.bias = np.zeros(n_classes)
        
        for i in range(self.n_iterations):
            # Forward pass
            linear = np.dot(X, self.weights) + self.bias
            y_pred = softmax(linear)
            
            # Gradients
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y, axis=0)
            
            # Update
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
        
        return self
    
    def predict(self, X):
        linear = np.dot(X, self.weights) + self.bias
        y_pred = softmax(linear)
        return np.argmax(y_pred, axis=1)

# Train and visualise
softmax_model = SoftmaxRegression()
softmax_model.fit(X_multi, y_multi_onehot)
y_pred_multi = softmax_model.predict(X_multi)

accuracy_multi = accuracy_score(y_multi, y_pred_multi)
print(f"Multiclass accuracy: {accuracy_multi:.3f}")

# Plot decision boundaries
def plot_multiclass_decision_boundary(X, y, model):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='viridis')
    plt.colorbar(scatter)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.title('Softmax Regression: Multiclass Decision Boundaries')
    plt.grid(True, alpha=0.3)

plt.figure(figsize=(8, 6))
plot_multiclass_decision_boundary(X_multi, y_multi, softmax_model)
plt.show()

print("\nSoftmax Regression:")
print("- Extends logistic regression to multiple classes")
print("- Each class gets a probability score")
print("- Predict the class with highest probability")

## Key Takeaways

- **Logistic regression** outputs probabilities via sigmoid function.
- **Cross-entropy loss** measures how well predicted probabilities match true labels.
- **Decision threshold** affects precision/recall tradeoff.
- **Linear decision boundaries** — assumes linear separability.
- **Regularisation** prevents overfitting in high-dimensional data.
- **Softmax** extends to multiclass problems.

## Exercises

1. **Sigmoid Derivative**: Derive dσ/dz. Show that σ′(z) = σ(z)·(1-σ(z)).

2. **Cross-Entropy Loss**: Why is cross-entropy better than MSE for classification?

3. **Decision Boundary**: For 2D features, what shape are logistic regression decision boundaries?

4. **Multiclass**: Implement one-vs-rest logistic regression for 3+ classes.

5. **Regularisation**: When would you use L1 vs L2 regularisation in logistic regression?

## Solutions

1. **Sigmoid Derivative**: σ(z) = 1/(1+e^{-z}), σ′(z) = σ(z)·(1-σ(z)). Proof: quotient rule.

2. **Cross-Entropy Loss**: MSE penalises wrong probabilities less, cross-entropy directly optimises log-likelihood.

3. **Decision Boundary**: Linear — hyperplanes in feature space.

4. **Multiclass**: Train K binary classifiers, each separating one class from rest. Predict class with highest confidence.

5. **Regularisation**: L1 for sparse models/feature selection. L2 for general regularisation when all features matter.